# October 2, 2026 scanner review

Frozen repository snapshot: `33b904efd8d0da9c86d83205c697645b29d0193a`. Times in data are UTC; subtract four hours for this session's Eastern times. Grain is one scan/ticker or one prospective contract observation. Repeated scan/ticker records are observations, not distinct trade opportunities.

- 80 stored scans from 9:17 to 15:57 ET; this is not a verified 16:00 closing snapshot.
- 35 prospective option observations, 55 follow-up quotes, zero verified quote timing. No option-return or fill claims follow from these records.
- JD chain not requested in 61/80 scans; DKNG 50/80; SOFI 59/80. These are allocation gaps, not provider failures.
- At commit a14a3d9a11a94f9fd28195f1d6d3bd16ec5be528 (10:37), SOFI 16 call bid .07/ask .08, delta .4691, volume 7368, no base rejection reasons, unknown quote timestamp: below default .10 price floor.
- DKNG bearish SECOND-WAVE at 10:57, stock 19.38; next saved 11:12 price18.945 (-2.24%). Chain SUCCESS, no preferred contract. This is observed stock movement, not an option return.
- JD 11:03 bearish TRIGGERED, stock25.97; no chain requested. This is an actionable data-coverage miss; no executable contract or missed option profit can be established.
- DKNG premarket 19.63/+3.32% implies approximately19.00 baseline. Regular-session snapshots use19.35. Code previously preferred any prior intraday bar over provider previousClose, including incomplete prior sessions.

Repairs: explicit previous-close provenance and completed-prior-session fallback; configurable priority allocation within the existing chain budget; price-cross labels separate from qualified setups; retained ranks, scores, baselines, volume, VWAP and contract rejection counts for future reviews. Historical data remains unchanged. Browser-specific price filters and personal fills are not repository evidence.


In [1]:
import json, subprocess, collections
revision = "33b904efd8d0da9c86d83205c697645b29d0193a"
def saved(name):
    return json.loads(subprocess.check_output(["git", "show", f"{revision}:data/{name}.json"]))
history = [s for s in saved("scan-history") if (s.get("generated_at") or "").startswith("2026-10-02")]
observations = [o for o in saved("option-observations") if o["market_date"] == "2026-10-02"]
print("Scans:", len(history), "from", history[0]["generated_at"], "to", history[-1]["generated_at"])
print("Option observations:", len(observations))
markouts = [m for o in observations for m in o["markouts"].values()]
print("Follow-up quotes:", len(markouts), "verified timing:", sum(bool(m.get("timing_verified")) for m in markouts))
for ticker in ["JD", "DKNG", "SOFI"]:
    rows = [r for s in history for r in s["candidate_state"] if r["ticker"] == ticker]
    print(ticker, dict(collections.Counter(r["chain_status"] for r in rows)))


Scans: 80 from 2026-10-02T13:17:21.559939+00:00 to 2026-10-02T19:57:03.393150+00:00
Option observations: 35
Follow-up quotes: 55 verified timing: 0
JD {'NOT_SCANNED': 61, 'SUCCESS': 19}
DKNG {'NOT_SCANNED': 50, 'SUCCESS': 30}
SOFI {'NOT_SCANNED': 59, 'SUCCESS': 21}
